# 1) Python fundamentals

This is the first subchapter of the introduction to Python. It builds the
language foundations everything else rests on: first, representing physical
quantities with primitive data types; then, structuring many values into
collections and automating decisions over them. An optional **Going deeper** box
shows how to set up a reproducible computational environment with modern tooling.

:::{admonition} Learning objectives
:class: tip
- Initialise an isolated computational environment using a modern dependency resolver.
- Manipulate primitive data types to represent physical environmental metrics.
- Automate repetitive environmental data filtering using loops and conditionals.
- Organise heterogeneous data using lists and dictionaries.
:::

:::{admonition} Takeaways
:class: danger
- The `uv` package manager standardises Python environments across operating systems, replacing fragile base-`pip` or Anaconda installs.
- Physical quantities are stored using specific primitive types, and the type dictates which operations are valid.
- Lists and dictionaries structure temporal and multidimensional data for automated processing.
- Control-flow structures let a script act on physical thresholds.
- Pure functions isolate logic, making pipelines modular and testable.
:::

:::{admonition} How this subchapter is organised
:class: note
The main text covers the essentials. Optional **Going deeper** boxes add advanced
or workflow material — open them if useful, skip them with no loss of continuity.
:::


:::{admonition} Computational thinking — decomposition and state representation
:class: note
A measurement is decomposed into typed pieces of state: a temperature is a
continuous number, a station name is text, a count is a whole number, a quality
flag is yes or no. Choosing the right type is the first modelling decision you make.
:::

## Variables and primitive types

A *variable* binds a name to a value, and every value has a **type**. The four
primitives you will use constantly are `float` (a real number), `int` (a whole
number), `str` (text), and `bool` (`True`/`False`).


In [1]:
temp_celsius = 15.5        # float: a continuous measurement
station_id = "NYON"        # str:   a text label
reading_count = 12         # int:   a whole count
is_valid = True            # bool:  a yes/no flag

for value in (temp_celsius, station_id, reading_count, is_valid):
    print(f"{value!r:>7}  ->  {type(value).__name__}")

   15.5  ->  float
 'NYON'  ->  str
     12  ->  int
   True  ->  bool


Notice the variable names. A name like `temp_celsius` or `discharge_m3s` encodes
the **physical meaning and unit** of the quantity; a name like `x` or `t` does
not. Naming quantities for what they physically are is the cheapest defence
against unit errors, which are among the most common bugs in scientific code.


## Basic operations

The arithmetic operators are `+ - * /` and `**` (power). We use them here for two
unit conversions: degrees Celsius to kelvin (the SI unit), and to Fahrenheit.


In [2]:
temp_kelvin = temp_celsius + 273.15
temp_fahrenheit = temp_celsius * 9 / 5 + 32
print(f"{temp_celsius} deg C = {temp_kelvin} K = {temp_fahrenheit} deg F")

15.5 deg C = 288.65 K = 59.9 deg F


Three built-in functions appear everywhere: `type()` reports a value's type,
`len()` gives the length of a collection or string, and `print()` displays output.


In [3]:
print(type(temp_celsius).__name__, "/", type(station_id).__name__)
print("station name has", len(station_id), "characters")

float / str
station name has 4 characters


:::{admonition} Going deeper — a reproducible environment with uv
:class: note dropdown
Base `pip` and Anaconda installs drift over time and collide between projects,
and compiled scientific dependencies (GDAL, PROJ) frequently fail to build. The
`uv` resolver creates fast, isolated, deterministic environments and behaves
identically across operating systems.

Install it once:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh
# macOS (Homebrew alternative)
brew install uv
```
```powershell
# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Create and activate an environment:

```bash
uv venv                              # create .venv
source .venv/bin/activate            # macOS / Linux (zsh, bash)
```
```powershell
.venv\Scripts\Activate.ps1           # Windows (PowerShell)
```

Add packages and run code with the environment's interpreter:

```bash
uv add numpy
uv run python my_script.py
```

Because `uv` records exact versions in a lockfile, the same commands reproduce
the same environment on your laptop and on a compute cluster.
:::


## Exercise — unit conversion

A prompt gives you a temperature reading of **15.5 °C**. Convert it to kelvin and
to Fahrenheit, round each result to two decimal places, and print a single
formatted sentence summarising all three values.

:::{admonition} Solution
:class: note dropdown
```python
temp_celsius = 15.5
temp_kelvin = round(temp_celsius + 273.15, 2)
temp_fahrenheit = round(temp_celsius * 9 / 5 + 32, 2)
print(f"{temp_celsius} deg C is {temp_kelvin} K and {temp_fahrenheit} deg F")
```
:::


## From single values to collections

A single reading is rarely the whole story. Environmental data arrives as
*collections* — a day of temperatures, a station's metadata. We now structure
those collections, automate decisions over them, and isolate logic into reusable
functions.

:::{admonition} Computational thinking — algorithmic thinking and pattern recognition
:class: note
Filtering a stream of readings is a repeatable rule: for each value, decide
whether it passes a test, and act accordingly. The flowchart shows that rule.
:::

```{mermaid}
flowchart TD
  A[raw readings] --> B{more values?}
  B -- no --> E[return clean list]
  B -- yes --> C[take next value]
  C --> D{value is an error code?}
  D -- yes --> B
  D -- no --> F[append to clean list]
  F --> B
```


## Lists

A *list* is an ordered, mutable collection, written with square brackets. Elements
are reached by **0-based** index; negative indices count from the end; a slice
`a:b` includes `a` and stops before `b`.


In [4]:
temps_c = [12.1, 13.4, 14.2, 11.8, 15.0]   # a run of hourly readings
print("first:", temps_c[0], " last:", temps_c[-1])
print("middle three:", temps_c[1:4])
temps_c[0] = 12.0          # lists are mutable
print("after edit:", temps_c)

first: 12.1  last: 15.0
middle three: [13.4, 14.2, 11.8]
after edit: [12.0, 13.4, 14.2, 11.8, 15.0]


## Dictionaries

A *dictionary* maps keys to values, ideal for heterogeneous metadata where each
field has a name rather than a position.


In [5]:
station = {"station_id": 1042, "name": "NYON", "elevation_m": 400}
print(station["name"], "sits at", station["elevation_m"], "m")
station["operational"] = True      # add a new field
print(station)

NYON sits at 400 m
{'station_id': 1042, 'name': 'NYON', 'elevation_m': 400, 'operational': True}


## Control flow

`if` / `elif` / `else` choose an action by testing conditions in order; the first
true branch wins. A `for` loop visits each element of a collection.


In [6]:
readings = [1.5, -2.0, 18.0, 7.5]
for t in readings:
    if t >= 15:
        band = "warm"
    elif t >= 0:
        band = "mild"
    else:
        band = "freezing"
    print(f"{t:>6} deg C -> {band}")

   1.5 deg C -> mild
  -2.0 deg C -> freezing
  18.0 deg C -> warm
   7.5 deg C -> mild


A `while` loop repeats as long as a condition holds — useful when the number of
steps is not known in advance.


In [7]:
# accumulate snowfall until 1 m water-equivalent is reached
daily_mwe = [0.2, 0.3, 0.1, 0.5, 0.4]
total, days = 0.0, 0
while total < 1.0 and days < len(daily_mwe):
    total += daily_mwe[days]
    days += 1
print(f"reached {total:.1f} m w.e. after {days} days")

reached 1.1 m w.e. after 4 days


## Pure functions

A *pure* function depends only on its inputs and returns a result without touching
anything outside itself. That isolation is what makes logic reusable and testable.


In [8]:
def mean(values):
    """Arithmetic mean of a non-empty list of numbers."""
    return sum(values) / len(values)

def drop_error_code(values, code=-9999.0):
    """Return a new list with all occurrences of an error code removed."""
    return [v for v in values if v != code]

raw = [12.1, 13.4, -9999.0, 14.2, 11.8]
clean = drop_error_code(raw)
print("clean:", clean, " mean:", round(mean(clean), 2))

clean: [12.1, 13.4, 14.2, 11.8]  mean: 12.88


## Exercise — filter and average

A prompt provides the raw array `[12.1, 13.4, -9999.0, 14.2, -9999.0, 11.8]`,
where `-9999.0` is a sensor error code. Initialise an empty list, loop through the
raw array, use an `if` statement to skip the error codes, append valid values to
the new list, and compute the mean with a custom function.

:::{admonition} Solution
:class: note dropdown
```python
raw = [12.1, 13.4, -9999.0, 14.2, -9999.0, 11.8]
clean = []
for value in raw:
    if value != -9999.0:
        clean.append(value)

def mean(values):
    return sum(values) / len(values)

print(clean, round(mean(clean), 2))
```
:::

More practice, including a code-review exercise on AI-generated code, is in the
companion exercise notebook.


## A code-review demonstration

Generated code can be subtly wrong in ways that raise no error. Consider an
AI-generated script meant to remove the `-9999.0` error codes by deleting them
*while iterating over the list* — a notorious trap.


In [9]:
# AI-generated (suspect): remove error codes while looping over the list
raw = [12.1, 13.4, -9999.0, 14.2, -9999.0, 11.8]
for value in raw:
    if value == -9999.0:
        raw.remove(value)
print("result on the given array:", raw)

result on the given array: [12.1, 13.4, 14.2, 11.8]


On this particular array it appears to work. That is exactly the danger: the
script is **silently data-dependent**. Run the same logic where two error codes
are *adjacent*:


In [10]:
raw2 = [12.1, -9999.0, -9999.0, 14.2]
for value in raw2:
    if value == -9999.0:
        raw2.remove(value)
print("result with adjacent error codes:", raw2)

result with adjacent error codes: [12.1, -9999.0, 14.2]


:::{admonition} Why it fails, and the fix
:class: note dropdown
Removing an element shifts every later element down one position, but the loop's
internal index still advances — so the element now occupying the vacated slot is
**skipped**. With adjacent error codes, the second one slides into the checked
position and is never tested, surviving into the "clean" data. No exception is
raised; it only manifests on some inputs.

Never mutate a collection while iterating it. Build a new list instead:

```python
clean = [v for v in raw2 if v != -9999.0]
```
:::


In [11]:
raw2 = [12.1, -9999.0, -9999.0, 14.2]
clean = [v for v in raw2 if v != -9999.0]
print("safe result:", clean)

safe result: [12.1, 14.2]


## Where this is going

You can represent physical quantities, structure them into collections, filter
and summarise them with pure Python. In the next subchapter we replace these
element-by-element loops with `numpy`, which applies operations to whole arrays at
once, far faster, and introduces the gridded datasets that dominate the Earth
sciences.
